### Ingest races.csv file

In [0]:
%run ../00-common/01-environment-config

In [0]:
%run ../00-common/02-bronze_helper

In [0]:
source_file = f"{landing_forlder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

races_schema = StructType(fields=[
    StructField("season", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("date", DateType(), True),
    StructField("circuitId", StringType(), True)
])

races_df = (spark.read.format("csv")
            .option("header", "true")
            .schema(races_schema)
            .option('mode', 'FAILFAST')
            .load(source_file)
            .select("*", "_metadata.file_path"))


In [0]:
display(races_df)

season,round,url,raceName,date,circuitId,file_path
1950,1,https://en.wikipedia.org/wiki/1950_British_Grand_Prix,british grand prix,1950-05-13,silverstone,dbfs:/Volumes/formula1/landing/files/races.csv
1950,2,https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix,monaco grand prix,1950-05-21,monaco,dbfs:/Volumes/formula1/landing/files/races.csv
1950,3,https://en.wikipedia.org/wiki/1950_Indianapolis_500,indianapolis 500,1950-05-30,indianapolis,dbfs:/Volumes/formula1/landing/files/races.csv
1950,4,https://en.wikipedia.org/wiki/1950_Swiss_Grand_Prix,swiss grand prix,1950-06-04,bremgarten,dbfs:/Volumes/formula1/landing/files/races.csv
1950,5,https://en.wikipedia.org/wiki/1950_Belgian_Grand_Prix,belgian grand prix,1950-06-18,spa,dbfs:/Volumes/formula1/landing/files/races.csv
1950,6,https://en.wikipedia.org/wiki/1950_French_Grand_Prix,french grand prix,1950-07-02,reims,dbfs:/Volumes/formula1/landing/files/races.csv
1950,7,https://en.wikipedia.org/wiki/1950_Italian_Grand_Prix,italian grand prix,1950-09-03,monza,dbfs:/Volumes/formula1/landing/files/races.csv
1951,1,https://en.wikipedia.org/wiki/1951_Swiss_Grand_Prix,swiss grand prix,1951-05-27,bremgarten,dbfs:/Volumes/formula1/landing/files/races.csv
1951,2,https://en.wikipedia.org/wiki/1951_Indianapolis_500,indianapolis 500,1951-05-30,indianapolis,dbfs:/Volumes/formula1/landing/files/races.csv
1951,3,https://en.wikipedia.org/wiki/1951_Belgian_Grand_Prix,belgian grand prix,1951-06-17,spa,dbfs:/Volumes/formula1/landing/files/races.csv


In [0]:
races_df_final = add_ingestion_metadata(races_df)

### Creating Delta Table

In [0]:
(
races_df_final.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(table_name)
)

In [0]:
%sql 
SELECT * from formula1.bronze.races limit 10

In [0]:
display(spark.sql("SHOW TABLES IN formula1.bronze"))